# 05 Inspect an Eelbrain TRF Pickle

This notebook opens one trusted local Eelbrain `BoostingResult` pickle and inventories its attributes without printing large arrays by default. It shows both public data attributes (including properties) and the internal fields actually stored in the pickled object.

> Only unpickle files that you created locally or obtained from a trusted source. Python pickle files can execute code while loading.

In [ ]:
from pathlib import Path
import inspect
import json
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from eelbrain import NDVar, load


def find_pipeline_dir(start=Path.cwd()):
    start = Path(start).resolve()
    candidates = [start, *start.parents, start / 'analysis' / 'trf_pipeline']
    for path in candidates:
        if (path / 'alice_eelbrain_main_experiment.py').exists():
            return path
    raise FileNotFoundError(f'Could not find alice_eelbrain_main_experiment.py from {start}')


PIPELINE_DIR = find_pipeline_dir()
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))

from alice_eelbrain_main_experiment import TRF_OPTIONS, alice

SUBJECT = '01'
MODEL = 'gammatone-8'
STATE = {'raw': '0.5-20', 'epoch': 'story-segments', 'inv': ''}

PICKLE_PATH = Path(
    alice.load_trf(MODEL, subject=SUBJECT, path_only=True, **STATE, **TRF_OPTIONS)
)
MANIFEST_PATH = PICKLE_PATH.with_name(PICKLE_PATH.name + '.manifest.json')

print(f'Subject: sub-{SUBJECT}')
print(f'Pickle: {PICKLE_PATH}')
print(f'Pickle exists: {PICKLE_PATH.exists()}')
print(f'Manifest: {MANIFEST_PATH}')
print(f'Manifest exists: {MANIFEST_PATH.exists()}')

In [ ]:
if not PICKLE_PATH.exists():
    raise FileNotFoundError(
        f'{PICKLE_PATH} does not exist. Run the single-subject or batch TRF notebook first.'
    )

result = load.unpickle(PICKLE_PATH)

print('Python type:', type(result))
print('Result:', result)
print('Stored field count:', len(vars(result)))

## Attribute inventory

Large `NDVar` and NumPy values are summarized by dimensions, shape, dtype, finite-value count, and basic statistics. The full object remains available through `result` and the `show_attribute()` helper below.

In [ ]:
def compact_repr(value, limit=240):
    text = repr(value).replace('\n', ' ')
    return text if len(text) <= limit else text[: limit - 3] + '...'


def array_statistics(array):
    array = np.asarray(array)
    out = {
        'shape': str(array.shape),
        'dtype': str(array.dtype),
        'finite': '',
        'min': '',
        'max': '',
        'mean': '',
    }
    if np.issubdtype(array.dtype, np.number):
        finite = np.isfinite(array)
        out['finite'] = f'{int(finite.sum())}/{array.size}'
        if finite.any():
            values = array[finite]
            out['min'] = float(values.min())
            out['max'] = float(values.max())
            out['mean'] = float(values.mean())
    return out


def summarize_value(name, value, source):
    row = {
        'name': name,
        'source': source,
        'python_type': f'{type(value).__module__}.{type(value).__qualname__}',
        'summary': compact_repr(value),
        'dimensions': '',
        'shape': '',
        'dtype': '',
        'finite': '',
        'min': '',
        'max': '',
        'mean': '',
        'error': '',
    }
    if isinstance(value, NDVar):
        row['dimensions'] = ', '.join(repr(dim) for dim in value.dims)
        row.update(array_statistics(value.x))
    elif isinstance(value, np.ndarray):
        row.update(array_statistics(value))
    return row


def inventory_public_attributes(obj):
    rows = []
    callable_rows = []
    for name in sorted(name for name in dir(obj) if not name.startswith('_')):
        try:
            value = getattr(obj, name)
        except Exception as exc:
            row = summarize_value(name, None, 'public attribute/property')
            row['error'] = repr(exc)
            rows.append(row)
            continue
        if callable(value):
            try:
                signature = str(inspect.signature(value))
            except (TypeError, ValueError):
                signature = '(signature unavailable)'
            callable_rows.append({'name': name, 'signature': signature})
        else:
            rows.append(summarize_value(name, value, 'public attribute/property'))
    return pd.DataFrame(rows), pd.DataFrame(callable_rows)


def inventory_stored_fields(obj):
    return pd.DataFrame(
        summarize_value(name, value, 'stored in pickle')
        for name, value in sorted(vars(obj).items())
    )


# Capture stored fields first: evaluating some public properties can cache
# derived values on the in-memory object that were not present in the pickle.
stored_fields = inventory_stored_fields(result)
public_attributes, public_methods = inventory_public_attributes(result)

In [ ]:
print(f'Public data attributes/properties: {len(public_attributes)}')
display(public_attributes)

print(f'Internal fields stored in pickle: {len(stored_fields)}')
display(stored_fields)

print(f'Public callable methods (listed separately): {len(public_methods)}')
display(public_methods)

## Inspect one attribute

Use `full=False` for a compact summary. Set `full=True` only when you intentionally want to print the underlying numeric array.

In [ ]:
def show_attribute(name, full=False):
    if not hasattr(result, name):
        raise AttributeError(f'{type(result).__name__} has no attribute {name!r}')
    value = getattr(result, name)
    print(f'{name}: {type(value)}')
    display(pd.DataFrame([summarize_value(name, value, 'selected attribute')]))
    if full:
        if isinstance(value, NDVar):
            display(value.x)
        elif isinstance(value, np.ndarray):
            display(value)
        else:
            display(value)
    else:
        display(value)
    return value


r = show_attribute('r')
h = show_attribute('h')

In [ ]:
sensor_names = list(result.r.sensor.names)
checks = {
    'model_is_gammatone_8': result.x == MODEL,
    'tstart_matches': result.tstart == TRF_OPTIONS['tstart'],
    'tstop_matches': result.tstop == TRF_OPTIONS['tstop'],
    'error_is_l1': result.error == 'l1',
    'basis_is_50_ms': result.basis == 0.050,
    'five_partitions': result.splits.n_partitions == 5,
    'cross_validated': result.splits.n_test == 1,
    'sixty_eeg_sensors': result.r.shape == (60,),
    'eight_gammatone_bands': result.h.shape[1] == 8,
    'veog_excluded': 'VEOG' not in sensor_names,
    'aud_excluded': 'AUD' not in sensor_names,
    'r_is_finite': bool(np.isfinite(result.r.x).all()),
    'h_is_finite': bool(np.isfinite(result.h.x).all()),
}
check_table = pd.DataFrame(
    [{'check': name, 'passed': passed} for name, passed in checks.items()]
)
display(check_table)
print(f'All technical checks passed: {all(checks.values())}')
print(f'r range: {float(np.min(result.r.x)):.6f} to {float(np.max(result.r.x)):.6f}')
print(f'partition_results stored: {result.partition_results is not None}')

## Inspect the cache manifest

The manifest is readable JSON. It records the cache key, estimator fingerprint, software versions, and input dependency fingerprints.

In [ ]:
if MANIFEST_PATH.exists():
    manifest = json.loads(MANIFEST_PATH.read_text())
    print('Manifest top-level keys:', list(manifest))
    display(manifest)
else:
    manifest = None
    print('No manifest found next to the pickle.')

## Optional summary export

Set `EXPORT_SUMMARIES = True` to write small CSV inventories. This exports summaries only, not the full `h` or `r` arrays.

In [ ]:
EXPORT_SUMMARIES = False
OUTPUT_DIR = Path('/Users/yanyuwoo/Data/Alice Comprehension/qc/trf_pickle_inspection')

if EXPORT_SUMMARIES:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    public_path = OUTPUT_DIR / f'sub-{SUBJECT}_{MODEL}_public_attributes.csv'
    stored_path = OUTPUT_DIR / f'sub-{SUBJECT}_{MODEL}_stored_fields.csv'
    checks_path = OUTPUT_DIR / f'sub-{SUBJECT}_{MODEL}_technical_checks.csv'
    public_attributes.to_csv(public_path, index=False)
    stored_fields.to_csv(stored_path, index=False)
    check_table.to_csv(checks_path, index=False)
    print('Wrote:', public_path)
    print('Wrote:', stored_path)
    print('Wrote:', checks_path)
else:
    print('EXPORT_SUMMARIES is False; no files were written.')